# Prototype FL deneyleri (Colab)

**Çalışma zamanı:** Üst menüden **Runtime → Change runtime type → GPU** (önerilir; CPU da çalışır, daha yavaş).

Aşağıdaki hücrelerden birinde projeyi getirin: **GitHub clone** veya **Drive’daki zip**.

## 1) Projeyi getir

**Seçenek A — GitHub:** `REPO_URL` değerini kendi repona göre değiştir.

In [ ]:
import os
import subprocess

CONTENT = "/content"
REPO_URL = "https://github.com/KULLANICI_ADIN/privproject.git"  # <-- kendi repo URL'in
BRANCH = "personalized-fl-murathan"  # veya main / personalized-fl
PROJECT_ROOT = os.path.join(CONTENT, "privproject")

os.chdir(CONTENT)
if not os.path.isdir(PROJECT_ROOT):
    subprocess.check_call(["git", "clone", REPO_URL, "privproject"])
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(["git", "checkout", BRANCH])
else:
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(["git", "pull"])

os.environ["PROJECT_ROOT"] = os.getcwd()
print("PROJECT_ROOT:", os.environ["PROJECT_ROOT"])

**Seçenek B — Drive:** Projeyi zipleyip Drive’a at; zip’i aç ve `privproject` klasörüne `cd` et (veya zip’i `/content` altına yükle ve `unzip`).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
# Drive’daki proje yolunu seçenek B için ayarla (clone kullanmıyorsan):
# import os
# os.environ["PROJECT_ROOT"] = "/content/drive/MyDrive/privproject"

## 2) Bağımlılıklar

Proje kökünde (`run_pfl.py` görünen dizin) olduğundan emin ol.

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
os.environ["PROJECT_ROOT"] = root
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=root,
)
print("pip OK:", root)

## 3) Ortam (isteğe bağlı)

Colab GPU’da genelde `cuda` kullanılır; kod `DEVICE` ile override edilebilir.

In [ ]:
import os

os.environ["DEVICE"] = "cuda"  # GPU yoksa: "cpu"
os.environ["NUM_CLIENTS"] = "10"
os.environ["NUM_ROUNDS"] = "2"
os.environ["EPOCH_SWEEP"] = "3,5,10"  # sadece --exp-mode epochs için
os.environ["EPOCHS"] = "5"  # augment / swa tek koşu için

print("DEVICE:", os.environ.get("DEVICE"))

## 4) Deneyler — üç ayrı komut

İstediğini çalıştır; çıktılar `outputs/metrics/experiments/...` altında.

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
subprocess.check_call(
    [sys.executable, "run_pfl.py", "--exp-mode", "epochs"],
    cwd=root,
)

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
subprocess.check_call(
    [sys.executable, "run_pfl.py", "--exp-mode", "augment"],
    cwd=root,
)

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
subprocess.check_call(
    [sys.executable, "run_pfl.py", "--exp-mode", "swa"],
    cwd=root,
)

## 5) Sonuçları indir

`outputs/` klasörünü zipleyip Drive’a kopyalayabilir veya Colab dosya panelinden indirirsin.

In [ ]:
import os
import subprocess
from google.colab import files

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
zip_path = "/content/pfl_outputs.zip"
if os.path.isfile(zip_path):
    os.remove(zip_path)
subprocess.check_call(
    ["zip", "-r", zip_path, "outputs", "-x", "*.pyc", "-x", "*__pycache__*"],
    cwd=root,
)
files.download(zip_path)